In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
# load all csvs
customers  = pd.read_csv("customers.csv")
orders     = pd.read_csv("orders.csv",     parse_dates=["order_created_at"])
deliveries = pd.read_csv("deliveries.csv", parse_dates=["dispatch_time", "delivery_completed_at"])
complaints = pd.read_csv("complaints.csv", parse_dates=["created_at"])
incidents  = pd.read_csv("incidents.csv")
drivers    = pd.read_csv("drivers.csv")
hubs       = pd.read_csv("hubs.csv")
vehicles   = pd.read_csv("vehicles.csv")
app_events = pd.read_csv("app_events.csv")

print(f"orders: {len(orders)} | deliveries: {len(deliveries)} | customers: {len(customers)} | drivers: {len(drivers)}")

In [ ]:
# check for missing values across all dataframes
for name, df in [("customers",customers),("orders",orders),("deliveries",deliveries),
                 ("complaints",complaints),("incidents",incidents),("drivers",drivers),
                 ("vehicles",vehicles),("app_events",app_events)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing):
        print(f"\n{name}:\n{missing.to_string()}")

In [ ]:
# normalise zones - strip whitespace, title case, fix Ctr -> Central
def norm_zone(s):
    return s.str.strip().str.title().replace("Ctr", "Central")

for df, cols in [
    (orders,     ["pickup_zone", "dropoff_zone"]),
    (customers,  ["home_zone"]),
    (drivers,    ["base_zone"]),
    (hubs,       ["zone"]),
    (app_events, ["zone_context"]),
    (deliveries, ["pickup_zone"])
]:
    for col in cols:
        if col in df.columns:
            df[col] = norm_zone(df[col])

print("zones after normalisation:", sorted(orders["pickup_zone"].unique()))

In [ ]:
# feature engineering - duration, failed flag, high value and override flags
deliveries["duration_hours"] = (
    (deliveries["delivery_completed_at"] - deliveries["dispatch_time"])
    / pd.Timedelta(hours=1)
)
deliveries.loc[deliveries["duration_hours"] < 0, "duration_hours"] = float("nan")
deliveries["failed"] = (deliveries["delivery_status"] == "Failed").astype(int)

del_ord = deliveries.merge(
    orders[["order_id", "pickup_zone", "order_value", "service_type"]], on="order_id", how="left"
)
del_ord["high_value"]    = (del_ord["order_value"] > del_ord["order_value"].quantile(0.75)).astype(int)
del_ord["override_used"] = (del_ord["manual_route_override_count"] > 0).astype(int)

print(del_ord[["failed", "high_value", "override_used"]].sum())
print(deliveries["duration_hours"].describe().round(2))

In [ ]:
# numpy stats summary across key delivery metrics
print("Delivery metric summary:\n")
for col in ["duration_hours", "fuel_or_charge_cost", "manual_route_override_count", "customer_rating_post_delivery"]:
    data = del_ord[col].dropna().values
    print(f"{col}:")
    print(f"  mean={np.mean(data):.2f}  std={np.std(data):.2f}  median={np.median(data):.2f}  95th={np.percentile(data, 95):.2f}")

In [ ]:
# zone failure rate bar chart
zone_fail = del_ord.groupby("pickup_zone")["failed"].mean().mul(100).reset_index()
zone_fail.columns = ["zone", "failure_rate"]
zone_fail = zone_fail.sort_values("failure_rate", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=zone_fail, x="zone", y="failure_rate", palette="RdYlGn_r")
plt.title("Delivery Failure Rate by Zone")
plt.xlabel("Zone")
plt.ylabel("Failure Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap + mann-whitney test to confirm failed deliveries take longer
num_cols = ["duration_hours", "manual_route_override_count", "fuel_or_charge_cost", "customer_rating_post_delivery", "failed"]
corr = del_ord[num_cols].dropna().corr()

failed_dur = deliveries[deliveries["failed"] == 1]["duration_hours"].dropna()
ok_dur     = deliveries[deliveries["failed"] == 0]["duration_hours"].dropna()
u, p = stats.mannwhitneyu(failed_dur, ok_dur)
print(f"Mann-Whitney U={u:.0f}, p={p:.2e} - failed deliveries take significantly longer")

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap - Delivery Metrics")
plt.tight_layout()
plt.show()

In [ ]:
# compensation cost breakdown by complaint type
comp_cost = complaints.groupby("complaint_type").agg(
    total=("complaint_id", "count"),
    avg_compensation=("compensation_amount", "mean"),
    total_compensation=("compensation_amount", "sum"),
    avg_resolution_days=("resolution_days", "mean")
).round(2).sort_values("total_compensation", ascending=False)

print(comp_cost)

In [ ]:
# bar chart - which complaint type costs the business the most
plt.figure(figsize=(10, 6))
sns.barplot(data=comp_cost.reset_index(), x="complaint_type", y="total_compensation", palette="Reds_r")
plt.title("Total Compensation Cost by Complaint Type")
plt.xlabel("Complaint Type")
plt.ylabel("Total Compensation (GBP)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# battery health vs incident count
veh_inc = (incidents
    .merge(deliveries[["delivery_id", "vehicle_id"]], on="delivery_id", how="left")
    .merge(vehicles[["vehicle_id", "battery_health_pct"]], on="vehicle_id", how="left")
    .groupby(["vehicle_id", "battery_health_pct"]).size().reset_index(name="incident_count"))

plt.figure(figsize=(10, 6))
sns.scatterplot(data=veh_inc, x="battery_health_pct", y="incident_count", color="orange", alpha=0.7)
sns.regplot(data=veh_inc, x="battery_health_pct", y="incident_count", scatter=False, color="black", line_kws={"linestyle": "--"})
plt.title("Battery Health vs Incident Count")
plt.xlabel("Battery Health (%)")
plt.ylabel("Incident Count")
plt.tight_layout()
plt.show()

In [ ]:
# top 15 drivers by failure rate - only drivers with 3+ deliveries
driver_perf = (del_ord
    .merge(drivers[["driver_id", "driver_rating", "training_score"]], on="driver_id", how="left")
    .groupby("driver_id")
    .agg(total=("delivery_id","count"), failures=("failed","sum"), avg_rating=("driver_rating","mean"))
    .query("total >= 3"))
driver_perf["failure_rate"] = (driver_perf["failures"] / driver_perf["total"] * 100).round(2)
driver_perf = driver_perf.sort_values("failure_rate", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=driver_perf.head(15).reset_index(), x="driver_id", y="failure_rate", palette="Reds_r")
plt.title("Top 15 Drivers by Failure Rate (min 3 deliveries)")
plt.xlabel("Driver ID")
plt.ylabel("Failure Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# app event failure rate by zone
app_fail = (app_events.groupby("zone_context")["success_flag"]
    .apply(lambda x: (x == 0).mean() * 100).reset_index(name="fail_rate")
    .sort_values("fail_rate", ascending=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=app_fail, x="zone_context", y="fail_rate", palette="PiYG")
plt.title("App Event Failure Rate by Zone")
plt.xlabel("Zone")
plt.ylabel("Failure Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# monthly order volume by service type
orders["month"] = orders["order_created_at"].dt.to_period("M").dt.to_timestamp()
monthly = orders.groupby(["month", "service_type"]).size().reset_index(name="count")

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly, x="month", y="count", hue="service_type", marker="o")
plt.title("Monthly Order Volume by Service Type")
plt.xlabel("Month")
plt.ylabel("Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()